In [16]:
from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [17]:
spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [18]:
df = spark.table("orders").na.replace("NULL", None)
df = df.na.fill("Unknown", "ShipRegion")

display(df.limit(3))

DataFrame[OrderID: string, CustomerID: string, EmployeeID: string, OrderDate: date, RequiredDate: date, ShippedDate: string, ShipVia: string, Freight: decimal(10,2), ShipName: string, ShipAddress: string, ShipCity: string, ShipRegion: string, ShipPostalCode: string, ShipCountry: string]

In [19]:

from pyspark.sql.functions import collect_list, collect_set, size

df_agg = (df.select(collect_list("ShipCity").alias("City_list"), 
                    collect_set("ShipCity").alias("City_set"))
            .select("*", size("City_list"), size("City_set")))
          
display(df_agg)

DataFrame[City_list: array<string>, City_set: array<string>, size(City_list): int, size(City_set): int]

In [20]:
from pyspark.sql.functions import collect_list, collect_set, size

df_agg = (df.groupBy("ShipRegion").agg(collect_list("ShipCity").alias("City_list"),
                                        collect_set("ShipCity").alias("City_set"))
           .select(size("City_list"), 
                   size("City_set"), "*"))

display(df_agg)

DataFrame[size(City_list): int, size(City_set): int, ShipRegion: string, City_list: array<string>, City_set: array<string>]